# TP2 — Named Entity Recognition with a CNN

Adapted from `cnn_classification.py`, which does **sentence** classification (sentiment).
NER is **token** classification, so four things change:

| | `cnn_classification.py` | this notebook |
|---|---|---|
| Data | CSV, one `review` + one `label` per row | CoNLL, one token + one tag per line |
| Output | 1 label per sentence | 1 label per token |
| Model | conv + **max-pool over time** -> 1 vector | conv with **same padding** -> 1 vector per token |
| Loss | `BCELoss` / `cross_entropy` on 2 classes | `cross_entropy` on 21 tags, ignoring padding |
| Metric | `sklearn.classification_report` | `seqeval` (entity-level P / R / F1) |

The embeddings are no longer random: they are initialised with the 6 models trained in TP1.

> The `argparse` block of the script is replaced by plain variables, since a notebook has no
> command line. Note the original script swaps its own arguments
> (`valid_file = args.test`, `test_file = args.valid`); that is fixed here.


## 0. Requirements


In [33]:
# on Google Colab, run this first
# %pip install torch pandas scikit-learn gensim seqeval tqdm


In [34]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 1. Configuration


In [35]:
DATA = Path('../TP_ISD2020/QUAERO_FrenchMed')
EMBEDDINGS_DIR = Path('../models')          # the models trained in TP1
MEDFULL_DIR = Path('../models_medfull')     # retrained on QUAERO_FrenchMed + MEDLINE

CORPUS = 'EMEA'                         # 'EMEA' (drug leaflets) or 'MEDLINE' (article titles)

train_file = DATA / CORPUS / f'{CORPUS}train_layer1_ID.conll'
valid_file = DATA / CORPUS / f'{CORPUS}dev_layer1_ID.conll'
test_file  = DATA / CORPUS / f'{CORPUS}test_layer1_ID.conll'

sequence_length = 128    # longest sentence in the corpus is 121 tokens, so nothing is truncated
max_vocab = 32000
embedding_size = 100     # must match the TP1 embeddings (dim = 100)

batch_size = 32
epochs = 30
lr = 1e-3
dropout = 0.25
grad_clip = 5
es_limit = 5             # early stopping patience

IGNORE_INDEX = -100      # tag id given to padding, ignored by the loss

## 2. Loading the CoNLL data

This replaces `load_and_preprocess_data`, which read a CSV of reviews.

Each line has 5 whitespace-separated columns: token index, token, two unused columns,
and the NER tag. An empty line separates two sentences.


In [36]:
def read_conll(path):
    """Read a CoNLL file as (sentences, tags), both lists of lists of strings."""
    sentences, tags = [], []
    tokens, labels = [], []

    for line in path.read_text(encoding='utf-8').splitlines():
        columns = line.split()
        if not columns:                  # empty line -> end of a sentence
            if tokens:
                sentences.append(tokens)
                tags.append(labels)
                tokens, labels = [], []
            continue
        tokens.append(columns[1])        # column 2 = the token
        labels.append(columns[4])        # column 5 = the NER tag

    if tokens:                           # last sentence, if the file has no trailing blank line
        sentences.append(tokens)
        tags.append(labels)

    return sentences, tags


train_sents, train_tags = read_conll(train_file)
valid_sents, valid_tags = read_conll(valid_file)
test_sents,  test_tags  = read_conll(test_file)

print(f'{CORPUS}: {len(train_sents)} train / {len(valid_sents)} dev / {len(test_sents)} test sentences')
print()
for token, tag in list(zip(train_sents[2], train_tags[2])):
    print(f'  {token:12s} {tag}')


EMEA: 706 train / 649 dev / 578 test sentences

  Qu           O
  ’            O
  est          O
  ce           O
  que          O
  Prialt       B-CHEM
  ?            O


### The tag set


In [37]:
# the tag vocabulary is built on the training set (dev and test use no other tag)
TAGS = sorted({t for tags in train_tags for t in tags})
tag2int = {t: i for i, t in enumerate(TAGS)}
int2tag = {i: t for t, i in tag2int.items()}
num_classes = len(TAGS)

print(f'{num_classes} tags:', ' '.join(TAGS))

counts = Counter(t for tags in train_tags for t in tags)
display(pd.Series(counts).sort_values(ascending=False).to_frame('train count').T)


21 tags: B-ANAT B-CHEM B-DEVI B-DISO B-GEOG B-LIVB B-OBJC B-PHEN B-PHYS B-PROC I-ANAT I-CHEM I-DEVI I-DISO I-GEOG I-LIVB I-OBJC I-PHEN I-PHYS I-PROC O


,O,B-CHEM,B-DISO,I-DISO,B-PROC,B-LIVB,I-CHEM,B-ANAT,I-PROC,B-PHYS,...,I-LIVB,B-DEVI,I-PHYS,I-ANAT,B-GEOG,B-PHEN,I-GEOG,I-OBJC,I-PHEN,I-DEVI
train count,12141,678,621,419,390,256,180,126,117,107,...,60,48,34,33,21,19,11,4,3,1


### Vocabulary and encoding

`pad_features` and `encode_review` from the script are kept, but they now pad the tags too:
tag padding gets `IGNORE_INDEX` so those positions are skipped by the loss and by the metrics.


In [38]:
def build_vocab(*sentence_lists, max_vocab=-1):
    """Build word2int from all splits, as the original script does."""
    words = [w for sents in sentence_lists for s in sents for w in s]
    counter = Counter(words)
    vocab = sorted(counter, key=counter.get, reverse=True)
    if max_vocab != -1:
        vocab = vocab[:max_vocab]
    int2word = dict(enumerate(vocab, 2))
    int2word[0] = '<PAD>'
    int2word[1] = '<UNK>'
    return {word: i for i, word in int2word.items()}


def encode(sentences, tags, word2int, seq_length=128):
    """Turn sentences and tags into two padded (n_sentences, seq_length) int arrays."""
    x = np.zeros((len(sentences), seq_length), dtype=np.int64)              # 0 = <PAD>
    y = np.full((len(sentences), seq_length), IGNORE_INDEX, dtype=np.int64)

    for i, (sentence, labels) in enumerate(zip(sentences, tags)):
        ids = [word2int.get(w, 1) for w in sentence][:seq_length]           # 1 = <UNK>
        lab = [tag2int[t] for t in labels][:seq_length]
        x[i, :len(ids)] = ids
        y[i, :len(lab)] = lab

    return x, y


vocab_index = build_vocab(train_sents, valid_sents, test_sents, max_vocab=max_vocab)
vocab_size = len(vocab_index)

train_x, train_y = encode(train_sents, train_tags, vocab_index, sequence_length)
valid_x, valid_y = encode(valid_sents, valid_tags, vocab_index, sequence_length)
test_x,  test_y  = encode(test_sents,  test_tags,  vocab_index, sequence_length)

print('Taille vocabulaire', vocab_size)
print('Feature Shapes:')
print('  Train set:     ', train_x.shape)
print('  Validation set:', valid_x.shape)
print('  Test set:      ', test_x.shape)


Taille vocabulaire 4590
Feature Shapes:
  Train set:      (706, 128)
  Validation set: (649, 128)
  Test set:       (578, 128)


In [39]:
trainset = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
validset = TensorDataset(torch.from_numpy(valid_x), torch.from_numpy(valid_y))
testset  = TensorDataset(torch.from_numpy(test_x),  torch.from_numpy(test_y))

trainloader = DataLoader(trainset, shuffle=True,  batch_size=batch_size)
valloader   = DataLoader(validset, shuffle=False, batch_size=batch_size)
testloader  = DataLoader(testset,  shuffle=False, batch_size=batch_size)


## 3. Initialising the embeddings with the TP1 models

In `cnn_classification.py` the embedding layer is `nn.Embedding(V, D)`, initialised randomly.
Here we build an embedding **matrix** from a TP1 model and copy it into that layer.

For fastText we load the `.bin`, so a word missing from the embedding vocabulary still gets a
vector built from its character n-grams. word2vec has no such fallback, and those words keep
their random initialisation.

Three extra models are included, named `medfull_*`. TP1 trained the medical embeddings on
`QUAERO_FrenchMed_traindev.ospl` alone (~52K tokens), which covers only 86% of the NER
vocabulary. `train_medfull_embeddings.py` retrains them on that corpus **plus** the
supplementary `MEDLINE_FR_tok.ospl` (~1.73M tokens in total), which should give both the
in-domain vocabulary of the medical corpus and the coverage of the press one. Run

```
python train_medfull_embeddings.py
```

to produce them; the notebook silently skips them if `../models_medfull` does not exist.

In [40]:
from gensim.models import Word2Vec
from gensim.models.fasttext import load_facebook_vectors

# the 6 models trained in TP1, the 3 retrained on the full medical corpus,
# plus a random baseline
EMBEDDINGS = {
    'random':               None,
    'med_w2v_cbow':         EMBEDDINGS_DIR / 'med_w2v_cbow.model',
    'med_w2v_skipgram':     EMBEDDINGS_DIR / 'med_w2v_skipgram.model',
    'med_ft_cbow':          EMBEDDINGS_DIR / 'med_ft_cbow.bin',
    'press_w2v_cbow':       EMBEDDINGS_DIR / 'press_w2v_cbow.model',
    'press_w2v_skipgram':   EMBEDDINGS_DIR / 'press_w2v_skipgram.model',
    'press_ft_cbow':        EMBEDDINGS_DIR / 'press_ft_cbow.bin',
    'medfull_w2v_cbow':     MEDFULL_DIR / 'medfull_w2v_cbow.model',
    'medfull_w2v_skipgram': MEDFULL_DIR / 'medfull_w2v_skipgram.model',
    'medfull_ft_cbow':      MEDFULL_DIR / 'medfull_ft_cbow.bin',
}

# skip the medfull models if train_medfull_embeddings.py has not been run yet
EMBEDDINGS = {k: v for k, v in EMBEDDINGS.items() if v is None or v.exists()}


def load_vectors(path):
    """Load a TP1 model as gensim KeyedVectors."""
    if path.suffix == '.bin':
        return load_facebook_vectors(str(path))   # fastText, can vectorise unknown words
    return Word2Vec.load(str(path)).wv            # word2vec


def build_embedding_matrix(path, word2int, dim=embedding_size):
    """Build a (vocab_size, dim) matrix, one row per word of our NER vocabulary."""
    kv = load_vectors(path)

    # words not covered keep a small random vector; <PAD> (row 0) stays at zero
    matrix = np.random.normal(0, 0.1, (len(word2int), dim)).astype(np.float32)
    matrix[0] = 0

    found = 0
    for word, i in word2int.items():
        if i < 2:                 # skip <PAD> and <UNK>
            continue
        try:
            matrix[i] = kv[word]  # for fastText this also works for unknown words
            found += 1
        except KeyError:
            pass

    coverage = found / (len(word2int) - 2)
    return matrix, coverage

How much of the NER vocabulary each TP1 model can cover:


In [41]:
coverages = {}
for name, path in EMBEDDINGS.items():
    if path is None:
        continue
    _, coverage = build_embedding_matrix(path, vocab_index)
    coverages[name] = f'{coverage:.1%}'

display(pd.Series(coverages).to_frame('vocabulary covered'))


,vocabulary covered
med_w2v_cbow,86.1%
med_w2v_skipgram,86.1%
med_ft_cbow,100.0%
press_w2v_cbow,58.2%
press_w2v_skipgram,58.2%
press_ft_cbow,100.0%


## 4. The model

`SentimentModelCNN` reduces a sentence to a single vector with
`F.max_pool1d(i, i.size(2))`, which destroys the position information. For NER we need one
prediction **per token**, so the max-pooling is removed and the convolutions use a padding
that preserves the sequence length. The output is `(batch, seq_length, num_classes)`
instead of `(batch, num_classes)`.

`Conv1d` replaces `Conv2d` here: with a `(K, D)` kernel over a `(1, W, D)` image the two are
equivalent, but `Conv1d` makes the length-preserving padding straightforward.


In [42]:
class NERModelCNN(nn.Module):

    def __init__(self, vocab_size, embedding_size, class_size, dropout=0.2,
                 pretrained=None, freeze=False):
        super(NERModelCNN, self).__init__()
        self.name = 'cnn'

        V = vocab_size
        D = embedding_size
        C = class_size
        Co = 100          # filters per kernel size
        Ks = [3, 4, 5]    # kernel sizes

        self.embed = nn.Embedding(V, D, padding_idx=0)
        if pretrained is not None:
            self.embed.weight.data.copy_(torch.from_numpy(pretrained))
            self.embed.weight.requires_grad = not freeze

        # padding=K//2 keeps the sequence length (an even K adds one step, trimmed below)
        self.convs = nn.ModuleList([nn.Conv1d(D, Co, K, padding=K // 2) for K in Ks])
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(len(Ks) * Co, C)

    def forward(self, x):
        W = x.size(1)
        x = self.embed(x)                                  # (N, W, D)
        x = x.transpose(1, 2)                              # (N, D, W)
        x = [F.relu(conv(x))[:, :, :W] for conv in self.convs]   # [(N, Co, W), ...]
        x = torch.cat(x, 1)                                # (N, len(Ks)*Co, W)
        x = x.transpose(1, 2)                              # (N, W, len(Ks)*Co)
        x = self.dropout(x)
        logit = self.fc1(x)                                # (N, W, C)  -> one tag per token
        return logit


## 5. Training and evaluation

The training loop keeps the structure of the script (per-epoch validation, early stopping on
the validation loss, best model kept). The loss is now computed over all tokens at once:
the logits are flattened to `(N*W, C)` and the targets to `(N*W,)`, and `ignore_index`
drops the padding positions.


In [43]:
def compute_loss(logits, targets):
    """Cross-entropy over every real token of the batch, padding excluded."""
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),   # (N*W, C)
        targets.reshape(-1),                   # (N*W,)
        ignore_index=IGNORE_INDEX,
    )


def token_accuracy(logits, targets):
    """Share of correctly tagged tokens, padding excluded."""
    predicted = logits.argmax(dim=-1)
    mask = targets != IGNORE_INDEX
    return (predicted[mask] == targets[mask]).float().mean().item()


def predict_tags(model, loader):
    """Run the model and return (true_tags, predicted_tags) as lists of tag sequences."""
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for feature, target in loader:
            feature, target = feature.to(device), target.to(device)
            predicted = model(feature).argmax(dim=-1)

            for gold_row, pred_row in zip(target.cpu().numpy(), predicted.cpu().numpy()):
                keep = gold_row != IGNORE_INDEX          # drop the padding
                y_true.append([int2tag[i] for i in gold_row[keep]])
                y_pred.append([int2tag[i] for i in pred_row[keep]])

    return y_true, y_pred


In [44]:
def train_model(pretrained=None, freeze=False, epochs=epochs, verbose=True):
    """Train one CNN and return it, together with its training history."""
    model = NERModelCNN(vocab_size, embedding_size, num_classes, dropout,
                        pretrained=pretrained, freeze=freeze).to(device)
    optim = Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    val_loss_min = torch.inf
    best_state = None
    es_trigger = 0

    epochloop = tqdm(range(epochs), desc='Training', leave=False)
    for e in epochloop:

        # training
        model.train()
        train_loss = train_acc = 0
        for feature, target in trainloader:
            feature, target = feature.to(device), target.to(device)

            optim.zero_grad()
            out = model(feature)
            loss = compute_loss(out, target)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optim.step()

            train_loss += loss.item()
            train_acc += token_accuracy(out, target)

        # validation
        model.eval()
        val_loss = val_acc = 0
        with torch.no_grad():
            for feature, target in valloader:
                feature, target = feature.to(device), target.to(device)
                out = model(feature)
                val_loss += compute_loss(out, target).item()
                val_acc += token_accuracy(out, target)

        history['train_loss'].append(train_loss / len(trainloader))
        history['train_acc'].append(train_acc / len(trainloader))
        history['val_loss'].append(val_loss / len(valloader))
        history['val_acc'].append(val_acc / len(valloader))
        epochloop.set_postfix_str(f"val loss {history['val_loss'][-1]:.3f}")

        # keep the best model, and stop if the validation loss stops improving
        if history['val_loss'][-1] <= val_loss_min:
            val_loss_min = history['val_loss'][-1]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            es_trigger = 0
        else:
            es_trigger += 1
            if es_trigger >= es_limit:
                if verbose:
                    print(f'Early stopped at epoch {e + 1}')
                break

    model.load_state_dict(best_state)
    return model, history


### A first run, with random embeddings

This is the baseline: the model of the original script, with its randomly initialised
embedding layer.


In [45]:
model, history = train_model(pretrained=None)
print(model)

y_true, y_pred = predict_tags(model, testloader)
print(classification_report(y_true, y_pred, digits=4, zero_division=0))


Early stopped at epoch 13
NERModelCNN(
  (embed): Embedding(4590, 100, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(100, 100, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): Conv1d(100, 100, kernel_size=(4,), stride=(1,), padding=(2,))
    (2): Conv1d(100, 100, kernel_size=(5,), stride=(1,), padding=(2,))
  )
  (dropout): Dropout(p=0.25, inplace=False)
  (fc1): Linear(in_features=300, out_features=21, bias=True)
)
              precision    recall  f1-score   support

        ANAT     0.2381    0.0459    0.0769       109
        CHEM     0.3441    0.1260    0.1844       762
        DEVI     0.2500    0.0132    0.0250        76
        DISO     0.2033    0.2490    0.2238       249
        GEOG     0.0000    0.0000    0.0000        14
        LIVB     0.8024    0.5751    0.6700       233
        OBJC     0.0000    0.0000    0.0000        49
        PHEN     0.0000    0.0000    0.0000        29
        PHYS     0.4545    0.0685    0.1190        73
        PROC     0.7750   

## 6. One model per embeddings type

The same CNN is now trained once per embeddings model: the 6 from TP1, the 3 retrained on the
full medical corpus, plus the random baseline.

In [46]:
results = {}
reports = {}

for name, path in EMBEDDINGS.items():
    matrix = None
    if path is not None:
        matrix, _ = build_embedding_matrix(path, vocab_index)

    model, history = train_model(pretrained=matrix, verbose=False)
    y_true, y_pred = predict_tags(model, testloader)

    results[name] = {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'epochs':    len(history['val_loss']),
    }
    reports[name] = classification_report(y_true, y_pred, digits=4, zero_division=0)
    print(f"{name:20s} P {results[name]['precision']:.4f}  "
          f"R {results[name]['recall']:.4f}  F1 {results[name]['f1']:.4f}")


Training:   0%|          | 0/30 [00:00<?, ?it/s]

random               P 0.4106  R 0.1954  F1 0.2648


med_w2v_cbow         P 0.3982  R 0.3359  F1 0.3644


med_w2v_skipgram     P 0.3959  R 0.3414  F1 0.3666


med_ft_cbow          P 0.3851  R 0.3172  F1 0.3479


press_w2v_cbow       P 0.4625  R 0.3930  F1 0.4249


press_w2v_skipgram   P 0.4668  R 0.3974  F1 0.4293


press_ft_cbow        P 0.4729  R 0.3738  F1 0.4175


### Results


In [47]:
scores = pd.DataFrame(results).T
scores[['precision', 'recall', 'f1']] = scores[['precision', 'recall', 'f1']].astype(float).round(4)
scores['epochs'] = scores['epochs'].astype(int)
display(scores.sort_values('f1', ascending=False))


,precision,recall,f1,epochs
press_w2v_skipgram,0.4668,0.3974,0.4293,19
press_w2v_cbow,0.4625,0.3930,0.4249,18
press_ft_cbow,0.4729,0.3738,0.4175,16
med_w2v_skipgram,0.3959,0.3414,0.3666,18
med_w2v_cbow,0.3982,0.3359,0.3644,18
med_ft_cbow,0.3851,0.3172,0.3479,18
random,0.4106,0.1954,0.2648,11


### Per-entity detail of the best model


In [48]:
best = scores['f1'].idxmax()
print(f'best: {best}\n')
print(reports[best])


best: press_w2v_skipgram

              precision    recall  f1-score   support

        ANAT     0.3378    0.2294    0.2732       109
        CHEM     0.5781    0.3740    0.4542       762
        DEVI     0.3333    0.0132    0.0253        76
        DISO     0.2118    0.4177    0.2811       249
        GEOG     0.4444    0.2857    0.3478        14
        LIVB     0.7820    0.7082    0.7432       233
        OBJC     0.0769    0.0204    0.0323        49
        PHEN     0.0000    0.0000    0.0000        29
        PHYS     0.3784    0.1918    0.2545        73
        PROC     0.5682    0.5482    0.5580       228

   micro avg     0.4668    0.3974    0.4293      1822
   macro avg     0.3711    0.2789    0.2970      1822
weighted avg     0.4966    0.3974    0.4244      1822

